# Week 14: LLM Fine-Tuning & Quantization
## Notebook 08: Prefix Tuning vs Prompt Tuning: Virtual Token Mechanics

### 1. Conceptual Intuition & Theoretical Background
While LoRA injects low-rank adapter matrices into linear projection layers, **Prompt Tuning** and **Prefix Tuning** freeze all original transformer weights and prepend trainable continuous virtual tokens.

**1. Prompt Tuning (Input-Level):**
Prepends $L$ virtual embedding vectors to the input sequence:
$$X = [P_1, P_2, \dots, P_L, x_1, x_2, \dots, x_N] \in \mathbb{R}^{(L+N) \times d}$$
- Only affects the very first embedding layer.

**2. Prefix Tuning (All-Layer Key-Value Tuning):**
Prepends virtual Key and Value prefixes across **every single attention block**:
$$\text{Attention}(Q, \tilde{K}, \tilde{V})$$
Where:
$$\tilde{K} = [P_K; K], \quad \tilde{V} = [P_V; V]$$
- Direct control over attention representations at every depth.
- Re-parameterized during training using an MLP ($P = \text{MLP}(E_\phi)$) to stabilize optimization.

---

### 2. Scratch Python Implementation: Multi-Layer Prefix Attention

In [ ]:
import numpy as np

class PrefixTunedAttentionScratch:
    def __init__(self, embed_dim=32, num_heads=4, prefix_len=5):
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.prefix_len = prefix_len
        
        # Frozen base transformer projections
        np.random.seed(42)
        self.W_q = np.random.normal(0, 0.1, (embed_dim, embed_dim))
        self.W_k = np.random.normal(0, 0.1, (embed_dim, embed_dim))
        self.W_v = np.random.normal(0, 0.1, (embed_dim, embed_dim))
        
        # Trainable Prefix Parameters for this layer (Keys and Values)
        # Parameterized across prefix_len virtual tokens
        self.prefix_k = np.random.normal(0, 0.05, (num_heads, prefix_len, self.head_dim))
        self.prefix_v = np.random.normal(0, 0.05, (num_heads, prefix_len, self.head_dim))
        
    def forward(self, x):
        """
        x: Input token representations (seq_len, embed_dim)
        """
        seq_len = len(x)
        
        # Compute standard Q, K, V from frozen base model
        Q = np.dot(x, self.W_q).reshape(seq_len, self.num_heads, self.head_dim).swapaxes(0, 1)
        K = np.dot(x, self.W_k).reshape(seq_len, self.num_heads, self.head_dim).swapaxes(0, 1)
        V = np.dot(x, self.W_v).reshape(seq_len, self.num_heads, self.head_dim).swapaxes(0, 1)
        
        # Prepend Prefix Keys and Values along sequence dimension
        # Shape becomes: (num_heads, prefix_len + seq_len, head_dim)
        K_prefixed = np.concatenate([self.prefix_k, K], axis=1)
        V_prefixed = np.concatenate([self.prefix_v, V], axis=1)
        
        # Scaled Dot-Product Attention: Q attends to both prefix and input sequence
        scores = np.matmul(Q, K_prefixed.swapaxes(-2, -1)) / np.sqrt(self.head_dim)
        weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        weights /= np.sum(weights, axis=-1, keepdims=True)
        
        context = np.matmul(weights, V_prefixed).swapaxes(0, 1).reshape(seq_len, self.embed_dim)
        return context

# Test Prefix-Tuned Layer
seq_len = 10
tokens = np.random.randn(seq_len, 32)

prefix_attn = PrefixTunedAttentionScratch(embed_dim=32, num_heads=4, prefix_len=6)
out = prefix_attn.forward(tokens)

print("--- Prefix Tuning Architecture ---")
print(f"Original Sequence Length:  {seq_len}")
print(f"Virtual Prefix Length:     {prefix_attn.prefix_len}")
print(f"Output Attention Shape:    {out.shape}")
print("Virtual prefix successfully modulates attention distributions across layers!")

### 3. Step-by-Step Code Walkthrough
1. **Prefix Concatenation**: Appends virtual parameters directly to the front of $K$ and $V$ matrices.
2. **Frozen Base Weights**: $W_q, W_k, W_v$ require zero gradient computation.
3. **Expressive Power**: Modulates attention steering at every layer.

---

### 4. Industry Applications & Trade-offs
- **Multi-Tenant Serving:** Storing thousands of task-specific prefixes in memory while sharing a single base foundation model.
- **Trade-off:** Virtual prefixes consume sequence context window length.

---

### 5. Top Interview Questions & Answers
**Q1: What is the primary difference between Prompt Tuning and Prefix Tuning?**
> **Answer:** Prompt Tuning prepends virtual token embeddings only at the input layer. Prefix Tuning prepends learnable Key and Value prefixes across all Transformer attention layers, providing greater expressiveness.

**Q2: Why is MLP reparameterization used when training Prefix Tuning?**
> **Answer:** Direct optimization of prefix embeddings is prone to unstable training dynamics. Projecting a smaller bottleneck embedding through a two-layer MLP ($P = W_2 \tanh(W_1 E)$) stabilizes convergence.